In [1]:
!pip install pdbparser biopython mygene openpyxl freesasa py3Dmol -q
import sys
!{sys.executable} -m pip install pymol-open-source -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 270.1/270.1 kB 5.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.9/161.9 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 40.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 48.8 MB/s eta 0:00:00


In [3]:
import pymol
from pymol import cmd
import matplotlib.colors as mcolors
import pandas as pd

# --- Parameters ---
pdb_file = 'Mouse STING homodimer(ranked_0).pdb'  # Ensure this file is in the same directory

# Define your residue color mapping { "residue_id": "hex_color" }
pathway_color = pd.DataFrame({
    "pathway": ["IFN", "UPR", "NFkB"],
    "color": ["#e3312a", "#7ac943", "#35b6e3"]})

excel_file = 'LOF_variants.csv'
residue_df_all = pd.read_csv(excel_file)

for pathway_name in ['IFN','NFkB','UPR']:

  # Filter rows where the group column contains any of the strings
  residue_df = residue_df_all[residue_df_all['group'].str.contains(pathway_name, na=False)].copy()

  # Get the color for the current pathway_name from pathway_color DataFrame
  current_pathway_color = pathway_color[pathway_color['pathway'] == pathway_name]['color'].iloc[0]
  residue_df['color'] = current_pathway_color

  residue_colors = dict(zip(residue_df['residue'].astype(str), residue_df['color']))
  output_png = "".join([pathway_name,"_LOF_3D.png"])


  # --- Initialize PyMOL ---
  # '-cq' runs PyMOL in command-line quiet mode
  pymol.finish_launching(['pymol', '-cq'])
  cmd.reinitialize()

  # Load structure and set base style
  cmd.load(pdb_file, 'prot')
  cmd.hide('everything', 'all')
  cmd.show('cartoon', 'prot')
  cmd.color('gray70', 'prot')

  # Visual clean-up (No shadows, white background)
  cmd.bg_color('white')
  cmd.set('ray_shadows', 0)
  cmd.set('depth_cue', 0)
  cmd.set('ray_opaque_background', 0)

  # --- Apply Sphere Coloring ---
  for resi, hex_color in residue_colors.items():
      # Define a unique name for the color in PyMOL
      color_name = f'color_{resi}'
      rgb = mcolors.to_rgb(hex_color)
      cmd.set_color(color_name, rgb)

      # Select and show spheres
      selection = f'resi {resi}'
      cmd.show('spheres', selection)
      cmd.color(color_name, selection)

  # Center the view on the highlighted spheres
  cmd.zoom('spheres', buffer=5)

  cmd.set_view([0.479985863,   -0.019433912,   -0.877059817,
      0.655858099,   -0.656033754,    0.373463154,
      -0.582640409,   -0.754482031,   -0.302139550,
      0.000000000,    0.000000000, -306.655059814,
      0.225017548,   -0.535675049,   -0.549560547,
    261.217132568,  352.092956543,  -20.000000000 ])

  # --- Render and Save ---
  cmd.ray(1200, 900)
  cmd.png(output_png, dpi=300)
  print(f'Successfully saved image to {output_png}')

Successfully saved image to IFN_LOF_3D.png
Successfully saved image to NFkB_LOF_3D.png
Successfully saved image to UPR_LOF_3D.png
